# statwarrant quick start

**What your statistical result warrants you to say.** statwarrant turns an estimate and its uncertainty
into a *claim contract* (what you may and may not say), shows the decision for every threshold of
practical importance, and can flag possible misreadings in a sentence you are about to write.

> **Experimental.** The statistics follow published methods and are tested against SciPy and
> statsmodels. The sentence checker is pattern-based and partial; its accuracy has not been measured on
> independently annotated sentences. Treat every warning as a prompt for your own review.

This notebook runs in Google Colab with nothing to install locally. Run the cells from top to bottom.

In [ ]:
# The version under public trial: the integrate/0.5.0 review branch.
# For an analysis you want to reproduce, replace the branch with a tag or commit.
REF = "integrate/0.5.0"
%pip install -q --force-reinstall --no-deps "statwarrant @ git+https://github.com/Haifawaeedd/statwarrant@{REF}"
%pip install -q numpy scipy

In [ ]:
import json
from importlib.metadata import distribution
import statwarrant as sw
source = json.loads(distribution("statwarrant").read_text("direct_url.json") or "{}")
print("statwarrant", sw.__version__, "commit", source.get("vcs_info", {}).get("commit_id", "?")[:7])

## 1. Declare before you look

The smallest effect size of interest (SESOI) is the smallest effect that would matter in practice. It
needs a stated reason. Write it down (and ideally register it) before seeing the result.

In [ ]:
plan = sw.declare(
    sesoi=3, sesoi_source="smallest score gain teachers said would change their practice",
    units="points", outcome="exam score", contrast="new method minus existing method",
    scope="first-year students, one university, one term",
)
res = plan.analyze(estimate=4.2, se=1.6, df=58)   # or a scipy / statsmodels result
print(res)

The result is statistically significant, yet the 3-point threshold is neither reached nor ruled out:
the data cannot say whether the effect matters in practice.

## 2. Every threshold at once

Someone else may use a different threshold. Two numbers answer every symmetric threshold.

In [ ]:
th = res.thresholds
print("an effect beyond ±d is established for every d below", th.met_below)
print("equivalence within ±d is established for every d above", th.equivalence_above)
for d in (0.5, 3, 8):
    print(f"d = {d}: {th.decision(d)}")

## 3. A real example from SciPy

Student's (1908) sleep data: extra hours of sleep for 10 patients under two drugs (paired). The
one-hour threshold is illustrative, not a clinical standard.

In [ ]:
from scipy import stats
drug1 = [0.7, -1.6, -0.2, -1.2, -0.1, 3.4, 3.7, 0.8, 0.0, 2.0]
drug2 = [1.9, 0.8, 1.1, 0.1, -0.1, 4.4, 5.5, 1.6, 4.6, 3.4]
sleep = sw.declare(sesoi=1, sesoi_source="illustrative only", units="hours",
                   outcome="extra hours of sleep", contrast="drug 2 minus drug 1",
                   scope="10 patients, Student (1908)").analyze(stats.ttest_rel(drug2, drug1))
print(sleep)

## 4. Check a sentence before you write it

The checker only warns. `conflict` means the sentence says something the result contradicts; `check`
means statwarrant could not verify a part, so review it yourself. No warning is **not** an approval:
the output lists the words it did not check.

In [ ]:
for sentence in [
    "There is no difference between the methods.",
    "The new method produced a large improvement.",
    "The new method works for all students.",
    "The difference was statistically significant (p = 0.011).",
]:
    print(res.check(sentence, record=False))
    print()

Try your own sentence about `res` or `sleep`:

In [ ]:
print(sleep.check("The two drugs had the same effect on sleep.", record=False))

## 5. How does this design behave?

Power, Type S (wrong sign) and exaggeration ratio under assumed true effects. Each assumption needs a
source. These describe the design, not the observed estimate.

In [ ]:
print(res.design([2, 3, 5], source="effects reported in two earlier studies"))

## 6. A report to keep

`to_markdown()` and `to_json()` record the plan, the tests, every threshold, the claims and what
statwarrant does not verify.

In [ ]:
print(res.to_markdown()[:1500])

## Tell us where it is wrong

The most useful feedback is a sentence the checker judged wrongly: a false warning, or a misreading it
missed. [Report a sentence](https://github.com/Haifawaeedd/statwarrant/issues/new?template=linter.yml) ·
[Give feedback](https://github.com/Haifawaeedd/statwarrant/issues/new?template=feedback.yml) ·
[Known issues](https://github.com/Haifawaeedd/statwarrant/blob/main/KNOWN_ISSUES.md).
Feedback is public: please do not paste confidential or unpublished text.